Notebook for fitting a simple neural network to a 2d SDF.

In [ ]:
# Run from the repository root, so that `import bcs` and the relative
# data/, configs/ and sdf_weights/ paths work wherever this notebook lives.
import os, sys
from pathlib import Path
_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'bcs').is_dir())
os.chdir(_root)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

In [ ]:
import os
from bcs.sdf_2d import *
import trimesh

def fit_in_bbox(vertices):
    
    min_vals = np.min(vertices, axis=0)
    max_vals = np.max(vertices, axis=0)
    
    # Shift the vertices by subtracting the minimum values for each axis
    vertices = vertices - min_vals

    ranges = max_vals - min_vals
    scale = 1.0 / np.max(ranges)
    print(scale)
    
    # Scale the shifted vertices so that the largest extent in any axis is 1
    vertices = vertices * scale
    
    vertices = 2 * vertices - 1

    new_min_vals = np.min(vertices, axis=0)
    new_max_vals = np.max(vertices, axis=0)

    vertices = vertices - (new_min_vals+new_max_vals)/2
    
    return vertices

In [ ]:
shape_name = 'eagle'

In [ ]:
OBJ_PATH = 'data/curves/'+shape_name+'_bdry.obj'  # <-- set to e.g. "curve.obj" or leave None to use demo circle polyline
mesh = trimesh.load(OBJ_PATH)
V = fit_in_bbox(mesh.vertices)

print(V[:,0].min(), V[:,0].max())
print(V[:,1].min(), V[:,1].max())
print(V[:,2].min(), V[:,2].max())

mesh.vertices=V
mesh.export(OBJ_PATH);

In [ ]:


# --- Main loading section ---
if OBJ_PATH is None or not os.path.exists(OBJ_PATH):
    print("OBJ not provided or not found. Using demo circle polyline.")
    poly = make_circle_polyline(radius=1.0, n=256)
    poly = np.vstack([poly, poly[0:1]])  # demo is explicitly closed
else:
    poly = load_polyline_obj(OBJ_PATH)
    print(f"Loaded closed polyline with {len(poly)} vertices (wrap last→first).")

# sample training data
#Might need to play with these parameters to get a good fit on shapes with more complex boundaries.
pts, sdf_vals = sample_training_points(poly, n_outside=3000, n_oncurve=15000, padding=0.3, ribbon_offset=0.1)
# fit network
model, device = fit_sdf_network(pts, sdf_vals, hidden=128, layers=4, lr=1e-3, epochs=10000, print_every=50)



In [ ]:
# plot results
plot_results( pts, sdf_vals, model=model, device=device, grid_res=300, polyline=poly, shape_name=shape_name)

In [ ]:

import torch
# Save model weights
WEIGHTS_PATH = "sdf_weights/curves/"+shape_name+".pth"

torch.save(model.state_dict(), WEIGHTS_PATH)
print(f"Model weights saved to {WEIGHTS_PATH}")


In [ ]:
import torch

WEIGHTS_PATH = "sdf_weights/curves/"+shape_name+".pth"


# Recreate the same model architecture
model, device = fit_sdf_network(
    pts[:1], sdf_vals[:1],  # dummy tiny dataset, just to build structure
    hidden=128, layers=4, epochs=0  # zero epochs, we’ll load weights
)

# Load weights
model.load_state_dict(torch.load(WEIGHTS_PATH, map_location=device))
model.eval()  # set to inference mode
print("Model weights loaded.")

In [ ]:
plot_results( pts, sdf_vals, model=model, device=device, grid_res=300, polyline=poly, shape_name=shape_name)